# Data Understanding
## Pengumpulan Data
Langkah pertama yaitu mengumpulkan data polutan udara NO₂ dan CO. Dataset ini mengambil dari platform satelit [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).


### Install Library & Autentikasi

Kita membutuhkan pustaka Python pendukung yaitu `openeo` untuk berkomunikasi dengan API Copernicus.

In [1]:
pip install openeo

^C


  Using cached openeo-0.51.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached shapely-2.1.2-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached xarray-2025.1.1-py3-none-any.whl.metadata (11 kB)
  Using cached pystac-1.15.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached oschmod-0.3.12-py2.py3-none-any.whl.metadata (10.0 kB)
  Using cached geopandas-1.1.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached wrapt-2.4.0-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
  Using cached pyproj-3.7.2-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached pystac_core-1.15.2-py3-none-any.whl.metadata (1.3 kB)
  Using cached pystac_ext_classification-2.0.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached pystac_ext_datacube-2.2.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached pystac_ext_eo-1.1.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached 

Note: you may need to restart the kernel to use updated packages.


Menghubungkan ke server openEO dan melakukan autentikasi menggunakan akun Copernicus Data Space.

In [ ]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Saat menjalankan baris di atas, akan muncul permintaan autentikasi:

```
Visit (link authentikasi) 📋 to authenticate.
✅ Authorized successfully
Authenticated using device code flow.
```


### Pengambilan Data NO₂ dan CO Area Wilayah Lamongan

Langkah berikutnya adalah menentukan wilayah spesifik. Titik koordinat batas wilayah Lamongan (Poligon) didapatkan menggunakan alat bantu pemetaan [geojson.io](https://geojson.io) dengan menggambar kotak di atas wilayah.

![Grafik Data](../img/geojson.png)

Koordinat yang didapatkan dimasukkan ke dalam variabel Area of Interest. Satelit Sentinel-5P kemudian diminta untuk mengambil data polutan berdasarkan _bounding box_ wilayah tersebut.

Mengingat satelit dapat merekam suatu area lebih dari satu kali dalam sehari, proses **agregasi temporal harian** diterapkan untuk mendapatkan satu nilai rata-rata per hari. Selanjutnya, dilakukan **agregasi spasial** guna merata-ratakan seluruh _grid_ di wilayah Lamongan menjadi satu representasi nilai tunggal.

### Memuat Data CO

In [ ]:
s5 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={
        "west": 112.3602,
        "south": -7.1799,
        "east": 112.4855,
        "north": -7.0641,
    },
    bands=["CO"],
)

### Memuat Data NO2

In [ ]:
s5 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={
        "west": 112.3602,
        "south": -7.1799,
        "east": 112.4855,
        "north": -7.0641,
    },
    bands=["NO2"],
)

Proses di jalankan batch job di server openEO, dan hasilnya dapat dipantau  melalui openEO Web Editor. [openEO editor](https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2). Setelah diproses oleh server, output akan otomatis diunduh dalam format **CSV**.

![Grafik Data](../img/openeo_editor.png)

### Hasil CSV
Disini memuat file CSV CO dan NO2 yang telah menggunakan pustaka Pandas. Dan disini kita hanya akan menampilkan 5 data teratas saja.

1. CO

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/CO_lamongan.csv")
df.head(5)

2. NO2

In [ ]:
df = pd.read_csv("../../data/No2_lamongan.csv")
df.head(5)

## Data Kosong (Missing Values) 

Ketidaklengkapan data atau _missing values_ merujuk pada situasi di mana titik-titik pengamatan tertentu tidak memiliki nilai ukur yang tercatat. Dalam konteks observasi satelit berbasis deret waktu, hilangnya data tersebut adalah hal yang lumrah. Pemicu utamanya berkisar dari halangan fisis seperti awan tebal yang menutupi area pandang sensor, hingga pola pergerakan orbit satelit yang menyebabkan absennya perekaman wilayah tersebut pada hari-hari tertentu. Mengenali rumpang data ini adalah prasyarat mutlak sebelum proses analisis dieksekusi.

Disini kita mengecek dua bentuk _missing values_:
1. **Tanggal yang Hilang**: Memastikan apakah ada urutan hari yang terlewat (bolong) dari rentang waktu awal hingga akhir (24 Agustus 2025 - 24 Agustus 2026).
2. **Data yang Hilang**: Memeriksa jumlah nilai polutan yang kosong (`NaN`) pada record tanggal yang sudah terekam.

### Tanggal Yang Hilang
1. CO

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/CO_lamongan.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal
start_date = "2025-08-24"
end_date   = "2026-08-24"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

2. NO₂

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/No2_lamongan.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal
start_date = "2025-08-24"
end_date   = "2026-08-24"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

### Data Yang Hilang

Kita juga akan mengecek jumlah baris data yang memiliki nilai konsentrasi polutan kosong .

1. CO

In [ ]:
df = pd.read_csv("../../data/CO_lamongan.csv")
missing_value = df['CO'].isna().sum()
print(missing_value)

Implementasi pada tools `Orange Data Mining`
```{image} ../img/missing_co.png
:alt: Grafik Data
:width: 100%
:align: center
```


2. NO₂

In [ ]:
df = pd.read_csv("../../data/No2_lamongan.csv")
missing_value = df['NO2'].isna().sum()
print(missing_value)

Implementasi pada tools `Orange Data Mining`

```{image} ../img/missing_no2.png
:alt: Grafik Data
:width: 100%
:align: center
```


## Outliers

Data ekstrem atau _outliers_ adalah nilai observasi yang berada sangat jauh dari rentang nilai wajar sebuah dataset. Untuk studi polusi udara, keberadaan angka ekstrem ini bisa merefleksikan kejadian nyata, seperti tiba-tiba naiknya emisi pabrik atau kebakaran lahan, ataupun sebatas kesalahan teknis (_error_) pada instrumen penginderaan jauh satelit.

Dalam upaya mendeteksi anomali tersebut di fase pemahaman data, kita memanfaatkan algoritma **Isolation Forest** dari modul `scikit-learn`. Mekanisme kerjanya adalah dengan membelah data secara acak untuk memisahkan observasi; logika utamanya adalah bahwa data yang anomali akan jauh lebih cepat terpisah ketimbang data yang normal. Dengan menentukan persentase toleransi pencilan (_contamination_) di angka 5%, model ini akan memetakan hasil. Tiap observasi yang ditandai dengan nilai prediksi `-1` dipastikan sebagai titik anomali.

1. CO

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../data/CO_lamongan.csv")
df_clean = df.dropna(subset=['CO']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['CO']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier:", jumlah_outlier)

Implementasi pada tools `Orange Data Mining`

```{image} ../img/outlier_co.png
:alt: Grafik Data
:width: 100%
:align: center
```


2. NO₂

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../data/No2_lamongan.csv")
df_clean = df.dropna(subset=['NO2']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['NO2']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier:", jumlah_outlier)

Implementasi pada tools `Orange Data Mining`

```{image} ../img/outlier_no2.png
:alt: Grafik Data
:width: 100%
:align: center
```